In [1]:
import os, logging

import polars as pl

from dotenv import load_dotenv

import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"

load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_READ_TOKEN") if os.getenv("HF_READ_TOKEN") else "" # type: ignore

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

# Configure logging levels to hide model-loading report
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

disable_progress_bar()

/home/spandanjit2005/Documents/26t2-codes/smart-mcq-colver-with-dl/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup (Run Before Attempting Questions) : 

# Use the following two fine-tuned sequence classification checkpoints:

# DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)
# RoBERTa: roberta-base (fine-tuned checkpoint)

# Label Mapping
# The models output logits for five labels corresponding to the answer options:
# Label ID Option
# 0             A
# 1             B
# 2             C
# 3             D
# 4             E

deberta_path = "microsoft/deberta-v3-small" 
roberta_path = "roberta-base"

# DeBERTa
tokenizer_deb = AutoTokenizer.from_pretrained(deberta_path)
model_deb = AutoModelForSequenceClassification.from_pretrained(deberta_path, num_labels=5)
model_deb.to(device).eval()

# RoBERTa
tokenizer_rob = AutoTokenizer.from_pretrained(roberta_path)
model_rob = AutoModelForSequenceClassification.from_pretrained(roberta_path, num_labels=5)
model_rob.to(device).eval()

label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

train_data = pl.read_csv("../../data/train.csv")
test_data = pl.read_csv("../../data/test.csv")

In [3]:
# Load the fine-tuned DeBERTa and RoBERTa models.

# For the prompt at row index 25, perform inference using each model 
# independently and apply Softmax to obtain class probabilities.

# Question 1:
# Which answer option receives the highest probability from the DeBERTa model, 
# and what is that probability?
# (answer format : eg - A, probability of A)

q1_2_data = train_data["prompt"][25]
inputs_deb = tokenizer_deb(
    q1_2_data, 
    max_length=512, 
    truncation=True,  
    return_tensors="pt"
)

with torch.no_grad():
    logits_deb = model_deb(**inputs_deb).logits
    
probs_deb = F.softmax(logits_deb, dim=-1)

max_prob_deb, max_idx_deb = torch.max(probs_deb, dim=-1)
q1_answer = label_map[max_idx_deb.item()] # type: ignore
q1_probability = max_prob_deb.item()

print(f"Q1 Answer: `{q1_answer}, {q1_probability:.4f}`")

Q1 Answer: `D, 0.2222`


In [4]:
# Using the same sample (row index 25), average the class probabilities from both models.

# Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

# Question 2:
# Which answer option receives the highest averaged probability after simple probability ensembling?

inputs_rob = tokenizer_rob(
    q1_2_data, 
    max_length=512, 
    truncation=True,  
    return_tensors="pt"
)

with torch.no_grad():
    logits_rob = model_rob(**inputs_rob).logits

probs_rob = F.softmax(logits_rob, dim=-1)

averaged_probs = (probs_deb + probs_rob) / 2

max_prob_avg, max_idx_avg = torch.max(averaged_probs, dim=-1)
q2_answer = label_map[max_idx_avg.item()] # type: ignore

print(f"Q2 Answer: {q2_answer}")

Q2 Answer: E


In [5]:
# Apply weighted probability averaging after Softmax using the following weights:
# DeBERTa: 0.70
# RoBERTa: 0.30

# Compute:
# P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

# Question 3:
# Which answer option is ranked first after weighted ensembling?

weighted_probs = (0.7 * probs_deb) + (0.3 * probs_rob)

max_prob_weighted, max_idx_weighted = torch.max(weighted_probs, dim=-1)
q3_answer = label_map[max_idx_weighted.item()] # type: ignore

print(f"Q3 Answer: {q3_answer}")

Q3 Answer: E


In [6]:
# Using the weighted ensemble probabilities from Q3, rank all five answer options.

# Write the final prediction exactly in Kaggle submission format.

# Question 4:
# What is the Top-3 prediction string for row index 25?
# Example : C A E

top3_indices = torch.argsort(weighted_probs, dim=-1, descending=True)[0][:3]
top3_labels = [label_map[idx.item()] for idx in top3_indices] # type: ignore
q4_answer = " ".join(top3_labels)

print(f"Q4 Answer: {q4_answer}")

Q4 Answer: E D A


In [7]:
# Run the weighted ensemble pipeline on every row of test.csv.

# Save the predictions in a file named submission.csv using the required Kaggle format:

# id,prediction

# where the prediction column contains the Top-3 ranked options separated by spaces.

# Question 5:
# Exactly how many prediction rows are present in the generated file (excluding the header)?

ids = []
predictions = []

for row in test_data.iter_rows(named=True):
    prompt_text = row["prompt"]
    row_id = row["id"]
    
    inputs_d = tokenizer_deb(
        prompt_text, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_d = model_deb(**inputs_d).logits
    p_deb = F.softmax(logits_d, dim=-1)
    
    inputs_r = tokenizer_rob(
        prompt_text, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_r = model_rob(**inputs_r).logits
    p_rob = F.softmax(logits_r, dim=-1)
    
    p_final = (0.7 * p_deb) + (0.3 * p_rob)
    
    top3_idx = torch.argsort(p_final, dim=-1, descending=True)[0][:3]
    pred_str = " ".join([label_map[idx.item()] for idx in top3_idx]) # type: ignore
    
    ids.append(row_id)
    predictions.append(pred_str)

submission_data = pl.DataFrame({
    "id": ids,
    "prediction": predictions
})

q5_answer = submission_data.height
print(f"Q5 Answer: {q5_answer}")

Q5 Answer: 500


In [8]:
# For the first 50 rows of test.csv, create two versions of every prompt:

# 1.Original prompt
# 2.Instruction-augmented prompt by prepending: 
#     "Answer the following multiple-choice question carefully:"

# Run inference using DeBERTa on both versions.
# Average the predicted probabilities from both passes.

# Question 6:
# How many of the first 50 rows produce a different Top-1 prediction after 
# applying Test-Time Augmentation?

q6_answer = 0
prefix_string = "Answer the following multiple-choice question carefully: "

for row in test_data.head(50).iter_rows(named=True):
    orig_prompt = row["prompt"]
    aug_prompt = prefix_string + orig_prompt
    
    inputs_orig = tokenizer_deb(
        orig_prompt, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_orig = model_deb(**inputs_orig).logits
    probs_orig = F.softmax(logits_orig, dim=-1)
    
    inputs_aug = tokenizer_deb(
        aug_prompt, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_aug = model_deb(**inputs_aug).logits
    probs_aug = F.softmax(logits_aug, dim=-1)
    
    probs_tta = (probs_orig + probs_aug) / 2
    
    top1_orig = torch.argmax(probs_orig, dim=-1).item()
    top1_tta = torch.argmax(probs_tta, dim=-1).item()
    
    if top1_orig != top1_tta:
        q6_answer += 1

print(f"Q6 Answer: {q6_answer}")

Q6 Answer: 0


In [9]:
# Process the first 100 rows of test.csv. And compare the Top-1 prediction from:
# 1. DeBERTa
# 2. Weighted Ensemble

# Question 7:
# How many rows have different Top-1 predictions?

# For the first 100 rows of test.csv, record the highest 
# class probability (confidence) predicted by:
# 1. DeBERTa
# 2. Weighted Ensemble

# For every row, compute:
# Confidence Gain = Ensemble Confidence−DeBERTa Confidence

# Question 8:
# How many rows have a positive confidence gain (greater than 0)?

q7_answer = 0
q8_answer = 0

for row in test_data.head(100).iter_rows(named=True):
    prompt_text = row["prompt"]
    
    inputs_d = tokenizer_deb(
        prompt_text, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_d = model_deb(**inputs_d).logits
    probs_d = F.softmax(logits_d, dim=-1)
    
    conf_d, max_idx_d = torch.max(probs_d, dim=-1)
    top1_d = max_idx_d.item()
    conf_d_val = conf_d.item()
    
    inputs_r = tokenizer_rob(
        prompt_text, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_r = model_rob(**inputs_r).logits
    probs_r = F.softmax(logits_r, dim=-1)
    
    probs_ens = (0.7 * probs_d) + (0.3 * probs_r)
    
    conf_ens, max_idx_ens = torch.max(probs_ens, dim=-1)
    top1_ens = max_idx_ens.item()
    conf_ens_val = conf_ens.item()
    
    # Question 7 Logic: Check for different Top-1 predictions
    if top1_d != top1_ens:
        q7_answer += 1
        
    # Question 8 Logic: Check for positive confidence gain
    conf_gain = conf_ens_val - conf_d_val
    if conf_gain > 0:
        q8_answer += 1

print(f"Q7 Answer: {q7_answer}")
print(f"Q8 Answer: {q8_answer}")

Q7 Answer: 86
Q8 Answer: 0


In [10]:
# For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:
# 1. DeBERTa alone
# 2. Weighted Ensemble

# Question 9:
# How many rows have at least one change in their ordered Top-3 ranking after ensembling?
# Examples:
# A C D vs. A D C

q9_answer = 0

for row in test_data.head(100).iter_rows(named=True):
    prompt_text = row["prompt"]
    
    inputs_d = tokenizer_deb(
        prompt_text, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_d = model_deb(**inputs_d).logits
    probs_d = F.softmax(logits_d, dim=-1)
    
    inputs_r = tokenizer_rob(
        prompt_text, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_r = model_rob(**inputs_r).logits
    probs_r = F.softmax(logits_r, dim=-1)
    
    probs_ens = (0.7 * probs_d) + (0.3 * probs_r)
    
    top3_idx_d = torch.argsort(probs_d, dim=-1, descending=True)[0][:3]
    top3_str_d = " ".join([label_map[idx.item()] for idx in top3_idx_d]) # type: ignore
    
    top3_idx_ens = torch.argsort(probs_ens, dim=-1, descending=True)[0][:3]
    top3_str_ens = " ".join([label_map[idx.item()] for idx in top3_idx_ens]) # type: ignore
    
    if top3_str_d != top3_str_ens:
        q9_answer += 1

print(f"Q9 Answer: {q9_answer}")

Q9 Answer: 86


In [11]:
# Using the Top-3 predictions generated by your weighted ensemble for 
# the first 100 validation samples, compute the MAP@3 score.

# Question 10:
# What is the final MAP@3 score? 
# (Round to 4 decimal places.)

ap_sum = 0.0
N_samples = 100

# Process first 100 rows train_data (serving as the validation set)
for row in train_data.head(N_samples).iter_rows(named=True):
    prompt_text = row["prompt"]
    true_answer = row["answer"]
    
    inputs_d = tokenizer_deb(
        prompt_text, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_d = model_deb(**inputs_d).logits
    probs_d = F.softmax(logits_d, dim=-1)
    
    inputs_r = tokenizer_rob(
        prompt_text, max_length=512, truncation=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits_r = model_rob(**inputs_r).logits
    probs_r = F.softmax(logits_r, dim=-1)
    
    probs_ens = (0.7 * probs_d) + (0.3 * probs_r)
    
    top3_idx_ens = torch.argsort(probs_ens, dim=-1, descending=True)[0][:3]
    top3_labels = [label_map[idx.item()] for idx in top3_idx_ens] # type: ignore
    
    if true_answer in top3_labels:
        rank = top3_labels.index(true_answer) + 1 
        ap_sum += 1.0 / rank

q10_answer = ap_sum / N_samples

print(f"Q10 Answer: {q10_answer:.4f}")

Q10 Answer: 0.2867
